In [1]:
import pandas as pd
import os, sys


REPO_ROOT = os.path.join(os.getcwd(), '../src')
if REPO_ROOT not in sys.path:
    sys.path.append(REPO_ROOT)

from src.data_load import TABLE_FILES, load_tables
from src.features import build_feature_table, feature_definitions
from src.id_utils import normalize_id
from src.infer import load_model_artifacts, predict_with_drivers, save_predictions, _prepare_inference_frame
from src.labeling import generate_weak_labels, labeling_thresholds
from src.memory_opt import (
    collect_garbage,
    log_memory_rss,
    optimize_dataframe_dtypes,
    optimize_table_dict,
    write_parquet_chunked,
)
from src.sampling import TrainSampleConfig, build_train_eval_samples, save_sample_outputs
from src.segments import compute_segment_kpis
from src.train import train_models

In [2]:
loaded = load_model_artifacts('../artifacts')
model = loaded["model"]
metadata = loaded.get("metadata", {})
feature_importance = loaded.get("feature_importance", {})
feature_columns = metadata.get("feature_columns", [])
class_labels = metadata.get("class_labels", ["AtRisk", "Healthy", "Warning"])
metrics = metadata.get("metrics", {})
selected_model = metadata.get("metrics", {}).get("selected_model", "loaded_artifact")
feature_importance

{'brand_id': 0.0,
 'window_size': 0.0,
 'window_size_days': 0.0,
 'recency_days_mean': 0.0,
 'recency_days_median': 0.0,
 'recency_days_p90': 0.0,
 'active_users': 0.0,
 'new_users': 0.0,
 'returning_users': 0.0,
 'dormant_share': 0.0,
 'total_events': 0.0,
 'total_logins': 0.0,
 'total_views': 0.0,
 'avg_events_per_active': 0.009982587208677137,
 'session_proxy_login_to_view': 0.0,
 'activity_events': 0.0,
 'activity_completed_events': 0.0,
 'activity_completion_rate': 0.0,
 'activity_points_sum': 0.0,
 'activity_points_per_active': 0.0,
 'activity_redeem_count': 0.0,
 'activity_redeem_rate': 0.0,
 'activity_redeem_per_active': 0.0,
 'activity_type_entropy': 0.0,
 'activity_type_top_share': 0.0,
 'activity_type_login_count': 0.0,
 'activity_type_login_completion_rate': 0.0,
 'activity_type_product_consume_uniquecode_count': 0.0,
 'activity_type_product_consume_uniquecode_completion_rate': 0.0,
 'activity_type_registration_count': 0.0,
 'activity_type_registration_completion_rate': 0.0

In [4]:
feature_df = pd.read_parquet('../outputs/labeled_feature_table.parquet')
feature_df

,brand_id,window_end_date,window_size,window_size_days,recency_days_mean,recency_days_median,recency_days_p90,active_users,new_users,returning_users,...,reward_efficiency_zscore,reward_efficiency_volatility,dormant_share_wow_pct,dormant_share_mom_pct,dormant_share_zscore,dormant_share_volatility,activity_points_per_active_wow_pct,label_health_score,label_health_class,label_health_class_int
0,c-vit,2021-01-10 00:00:00+00:00,30d,30.0,53.956000,48.0,85.1,108.0,59.0,49.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,100.000000,Healthy,2
1,c-vit,2021-01-17 00:00:00+00:00,30d,30.0,59.927345,55.0,92.0,102.0,50.0,52.0,...,0.000000,0.000000,0.010815,0.000000,0.000000,0.000000,0.0,100.000000,Healthy,2
2,c-vit,2021-01-24 00:00:00+00:00,30d,30.0,65.628422,62.0,99.0,79.0,33.0,46.0,...,0.577350,0.005516,0.036687,0.000000,1.127632,0.024639,0.0,63.813725,Warning,1
3,c-vit,2021-01-31 00:00:00+00:00,30d,30.0,71.037179,69.0,106.0,72.0,32.0,40.0,...,0.500000,0.004773,0.011919,0.000000,1.057990,0.028098,0.0,92.000000,Healthy,2
4,c-vit,2021-02-07 00:00:00+00:00,30d,30.0,77.104193,76.0,112.4,75.0,37.0,38.0,...,-1.656485,0.011000,-0.003295,0.056894,0.779304,0.026892,0.0,56.952381,Warning,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1627,see-chan,2025-12-31 00:00:00+00:00,90d,90.0,149.836440,93.0,290.0,84402.0,63874.0,20528.0,...,-1.452220,0.000020,-0.031947,-0.079728,-1.149658,0.086264,0.0,83.308878,Healthy,2
1628,see-chan,2026-01-07 00:00:00+00:00,90d,90.0,156.413755,100.0,297.0,84001.0,62868.0,21133.0,...,-1.109452,0.000021,0.006233,-0.059530,-0.895968,0.075219,0.0,100.000000,Healthy,2
1629,see-chan,2026-01-14 00:00:00+00:00,90d,90.0,162.605491,99.0,304.0,77239.0,57184.0,20055.0,...,-0.868753,0.000021,0.081855,0.048433,0.501748,0.051372,0.0,81.266101,Healthy,2
1630,see-chan,2026-01-21 00:00:00+00:00,90d,90.0,169.006717,105.0,311.0,75659.0,55163.0,20496.0,...,-0.677378,0.000018,0.018437,0.073249,1.357262,0.038272,0.0,96.000000,Healthy,2


In [5]:
input_infer = _prepare_inference_frame(feature_df, feature_columns)
input_infer

,brand_id,window_size,window_size_days,recency_days_mean,recency_days_median,recency_days_p90,active_users,new_users,returning_users,dormant_share,...,reward_efficiency_zscore,reward_efficiency_volatility,dormant_share_wow_pct,dormant_share_mom_pct,dormant_share_zscore,dormant_share_volatility,activity_points_per_active_wow_pct,window_end_ordinal,window_end_month,window_end_week
0,c-vit,30d,30.0,53.956000,48.0,85.1,108.0,59.0,49.0,0.856000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,737800,1.0,1.0
1,c-vit,30d,30.0,59.927345,55.0,92.0,102.0,50.0,52.0,0.865258,...,0.000000,0.000000,0.010815,0.000000,0.000000,0.000000,0.0,737807,1.0,2.0
2,c-vit,30d,30.0,65.628422,62.0,99.0,79.0,33.0,46.0,0.897001,...,0.577350,0.005516,0.036687,0.000000,1.127632,0.024639,0.0,737814,1.0,3.0
3,c-vit,30d,30.0,71.037179,69.0,106.0,72.0,32.0,40.0,0.907692,...,0.500000,0.004773,0.011919,0.000000,1.057990,0.028098,0.0,737821,1.0,4.0
4,c-vit,30d,30.0,77.104193,76.0,112.4,75.0,37.0,38.0,0.904701,...,-1.656485,0.011000,-0.003295,0.056894,0.779304,0.026892,0.0,737828,2.0,5.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1627,see-chan,90d,90.0,149.836440,93.0,290.0,84402.0,63874.0,20528.0,0.502370,...,-1.452220,0.000020,-0.031947,-0.079728,-1.149658,0.086264,0.0,739616,12.0,1.0
1628,see-chan,90d,90.0,156.413755,100.0,297.0,84001.0,62868.0,21133.0,0.505501,...,-1.109452,0.000021,0.006233,-0.059530,-0.895968,0.075219,0.0,739623,1.0,2.0
1629,see-chan,90d,90.0,162.605491,99.0,304.0,77239.0,57184.0,20055.0,0.546879,...,-0.868753,0.000021,0.081855,0.048433,0.501748,0.051372,0.0,739630,1.0,3.0
1630,see-chan,90d,90.0,169.006717,105.0,311.0,75659.0,55163.0,20496.0,0.556962,...,-0.677378,0.000018,0.018437,0.073249,1.357262,0.038272,0.0,739637,1.0,4.0


In [6]:
model.predict(input_infer)
# input_infer

array(['Healthy', 'Healthy', 'Warning', ..., 'Healthy', 'Healthy',
       'Warning'], shape=(1632,), dtype=object)

## Loaded from static output predicted

In [ ]:
predict_df = pd.read_csv('../outputs/predicted_df.csv')
predict_df['window_end_date'] = pd.to_datetime(predict_df['window_end_date'])
predict_df = predict_df[predict_df['window_end_date'] == '2025-12-31']
predict_df

## Layer 1
| Feature                | บทบาท                             | Impact ใน Layer 1           |
| ---------------------- | --------------------------------- | --------------------------- |
| predicted_health_score | Continuous score (0–1 หรือ 0–100) | ใช้จัด ranking และ severity |
| prob_Healthy           | Model confidence                  | ความมั่นใจว่าปกติ           |
| prob_Warning           | Early risk signal                 | อยู่โซนกลาง เริ่มมีสัญญาณ   |
| prob_AtRisk            | Churn / collapse risk             | สัญญาณอันตรายสูง            |


In [ ]:
layer1_df = predict_df[['window_end_date', 'window_size', 'predicted_health_score', 'prob_Healthy', 'prob_Warning', 'prob_AtRisk']].copy()
layer1_df.head()

## Layer 2
| Feature                          | มิติ (Symptom Dimension) | Impact ต่อ Layer 2 (Root Cause Explanation)           |
| -------------------------------- | ------------------------ | ----------------------------------------------------- |
| active_users_wow_pct             | Growth Momentum          | ถ้าติดลบ แปลว่า acquisition หรือ activation กำลังชะลอ |
| new_user_share                   | Acquisition Structure    | ถ้าต่ำ แปลว่า funnel ต้นน้ำอ่อนแรง                    |
| gmv_net_wow_pct                  | Revenue Momentum         | ถ้าหดตัว แปลว่า conversion หรือ spending ลดลง         |
| activity_completion_rate         | Engagement Quality       | ต่ำ = ผู้ใช้ทำ activity ไม่จบหรือไม่สนใจ              |
| activity_completion_rate_wow_pct | Engagement Momentum      | ลดลงต่อเนื่อง = engagement fatigue                    |
| activity_points_per_active       | Engagement Depth         | ต่ำ = interaction ต่อคนตื้น                           |
| activity_type_entropy            | Engagement Diversity     | ต่ำ = behavior ซ้ำ ๆ ไม่หลากหลาย                      |
| dormant_share                    | Retention Level          | สูง = ฐานผู้ใช้กำลังเสื่อม                            |
| dormant_share_wow_pct            | Retention Momentum       | เพิ่มเร็ว = churn wave กำลังมา                        |
| recency_days_mean                | User Freshness           | สูงขึ้น = ผู้ใช้กลับมาห่างขึ้น                        |
| rfm_transition_down_share        | Segment Downgrade Risk   | สูง = กลุ่มลูกค้ากำลัง drop tier                      |
| gmv_per_active                   | Revenue Intensity        | ต่ำ = รายได้ต่อ active user อ่อน                      |
| transactions_per_active          | Purchase Frequency       | ต่ำ = พฤติกรรมซื้อห่าง                                |
| reward_efficiency                | Monetisation Efficiency  | ต่ำ = burn reward แต่ไม่ convert เป็น GMV             |


In [ ]:
layer2_df = predict_df[['window_end_date', 'window_size', 'active_users_wow_pct', 'new_user_share', 'gmv_net_wow_pct', 'activity_completion_rate',
                        'activity_completion_rate_wow_pct', 'activity_points_per_active', 'activity_type_entropy',
                        'dormant_share', 'dormant_share_wow_pct', 'recency_days_mean', 'rfm_transition_down_share',
                        'gmv_per_active', 'transactions_per_active', 'reward_efficiency', 'target_segments', 'drivers']].copy()
layer2_df.head()

## Merge Layer 1 and Layer 2

In [ ]:
df = pd.merge(layer1_df, layer2_df, on=['window_end_date', 'window_size'], how='inner')
df